# Tutorial 7.3: Simulation of Cell-Type-Specific Incomplete Registration in COAD

Building on the COAD adjacent-section registration in Tutorial 7.1, this tutorial creates a controlled cell-type-specific incomplete-registration benchmark. The unfiltered scSLAT RNA-on-CODEX reference transfers an RNA profile to every CODEX coordinate, providing complete source RNA alongside measured 16-plex CODEX protein without assuming direct physical RNA-protein pairing.

All CODEX protein profiles in a selected cell type (e.g., fibroblasts or CD4+ T cells) are withheld while RNA remains available at the same coordinates. This structured target gap tests whether PRISM can preserve spatial organization and predict protein in a biologically coherent compartment; the pre-masking protein measurements provide location-matched ground truth.


In [ ]:
# 0. Environment and imports
from pathlib import Path

import scanpy as sc
import PRISM
from PRISM import (plot_imputation_metric_boxplot, compute_similarity_prior, plot_per_protein_correlations,
                   plot_prism_imputation_spatial, preprocess_omics, prism_eval_and_save,
                   run_clustering_eval_plot, select_best_device, set_prism_plot_style, show_real_missing,
                   simulate_celltype_missing)
set_prism_plot_style()

### Load data

`adata_matched.h5ad` provides the unfiltered scSLAT RNA transfer in CODEX coordinate order, and the original CODEX object supplies complete measured protein profiles. Together they define the complete reference used for controlled masking.


In [ ]:
DEVICE = select_best_device()

DATASET_DIR = Path("Datasets") / "COAD"
SOURCE_H5AD = DATASET_DIR / "adata_matched.h5ad"
TARGET_H5AD = DATASET_DIR / "adata_codex.h5ad"
RESULTS_DIR = Path("Results") / "Tutorial7_3_COAD_celltype"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRIOR_PATH = RESULTS_DIR / "COAD_RNA_AOT_32k.npz"
RUN_PREFIX = "COAD_PRISM"

adata_source_raw = sc.read_h5ad(SOURCE_H5AD)
adata_target_raw = sc.read_h5ad(TARGET_H5AD)
adata_source_raw.var_names_make_unique()
adata_target_raw.var_names_make_unique()


In [ ]:
adata_target_raw.obs['annotation']

The original CODEX AnnData object stores data-provided cell-type annotations in `obs["annotation"]`. These labels can be inspected here and used to select one or more populations through `missing_celltypes` in the next cell for cell-type-specific unregistration simulation.


### Simulating cell-type-specific unregistration

`simulate_celltype_missing` designates all `Fibroblast` CODEX protein profiles as target-unregistered while retaining the transferred RNA source at the same coordinates. `missing="0"` denotes the held-out target cells and `missing="1"` the observed RNA-protein pairs, modelling cell-type-specific technical target absence rather than biological protein loss.


In [ ]:
adata_target_sim, missing_indices, observed_indices = simulate_celltype_missing(
                                                         adata_target_raw, annotation_key="annotation",
                                                         missing_celltypes=["Fibroblast"], missing_key="missing",
                                                         inplace=False)

adata_source_raw.obs["missing"] = "1"
show_real_missing(adata_target_sim, spatial_key="spatial", label_key="missing", plot=True, figsize=(4, 4),
                  s=0.01, title="Cell-type-specific protein missingness")
print(f"Simulated protein-missing cells: {len(missing_indices)}/{adata_target_sim.n_obs}")

In [ ]:
# Preprocessing source (CODEX protein) and target (RNA)
adata_source, _ = preprocess_omics(adata_source_raw, modality="RNA", missing_key="missing", min_cells=10,
                                   hvgs=2048, data_role="source", compute_pca=False, save_raw_eval=False)

adata_target, _ = preprocess_omics(adata_target_sim, modality="ADT", missing_key="missing",
                                   data_role="target", compute_pca=False, save_raw_eval=True)

print("RNA shape after preprocessing:", adata_source.shape)
print("CODEX protein shape after preprocessing:", adata_target.shape)

### Constructing the RNA similarity prior

Complete RNA profiles define the source spatial-niche prior, which retrieves observed RNA-CODEX locations with related local RNA context for protein-unregistered fibroblast queries. The 300 AOT environments are constructed in 4,096-cell blocks to keep this large dataset memory-bounded.


In [ ]:
# Compute RNA similarity prior
distance_matrix, _ = compute_similarity_prior(adata_source, adata_target, PRIOR_PATH, device=DEVICE,
                                              covet_k_spatial=32, covet_gene_num=64, aot_k_env=300,
                                              aot_chunk_size=4096, spatial_key="spatial", missing_key="missing",
                                              batch_key="batch" if "batch" in adata_source.obs else -1,
                                              evaluate_prior=False)

In [ ]:
# Constructing spatial graphs
PRISM.Cal_Spatial_Net(adata_source, rad_cutoff=23)
PRISM.Stats_Spatial_Net(adata_source)
PRISM.Cal_Spatial_Net(adata_target, rad_cutoff=23)
PRISM.Stats_Spatial_Net(adata_target)

### Training PRISM

PRISM integrates complete RNA, the observed RNA-CODEX pairs outside the held-out fibroblast population, spatial graphs and the RNA-derived prior to predict protein at target-unregistered locations. The learned representation supports spatial-domain analysis, while the protein decoder supplies the missing-omics prediction.


In [ ]:
# Train PRISM with complete RNA and simulated incomplete CODEX protein
adata_source_out, adata_target_out = PRISM.train_PRISM(adata_source, adata_target, distance_matrix,
                                                       k_top=5, n_epochs=800, lr=6e-4,
                                                       output_dir=str(RESULTS_DIR), file_prefix=RUN_PREFIX,
                                                       device=DEVICE, patience=15, min_epochs=20,
                                                       center_drop_rate=0.1, noise=0.0,
                                                       interaction_pca=True)

### Task 1: Spatial-domain identification

In [ ]:
# Identify domains from the PRISM embedding and evaluate them post hoc
adata_clustered, domain_metrics = run_clustering_eval_plot(adata_source_out, emb_key="PRISM_emb_base", 
                                                           label_key="spatial_cluster", cluster_key="PRISM_mclust",
                                                           n_clusters=5, s=1, use_pca=True, align_labels=True,
                                                           aligned_key="PRISM_domain", dataset_name="COAD")

### Task 2: CODEX protein imputation

Task 2 evaluates protein imputation at the simulated fibroblast target-unregistered cells against their pre-masking raw profiles. Overall performance is reported as the mean +/- s.d. of feature-wise Pearson correlation coefficient (PCC), Spearman correlation coefficient (SPCC) and mean squared error (MSE) across CODEX proteins.


In [ ]:
# Quantitatively evaluate CODEX protein imputation in simulated missing cells
imputation_results = prism_eval_and_save(truth_adata=adata_target_out, adata=adata_target_out,
                                         save_path=str(RESULTS_DIR), first_name=RUN_PREFIX,
                                         missing_indices=missing_indices, save_files=False)

In [ ]:
# Display per-protein imputation metrics
per_protein_correlation_figures = plot_per_protein_correlations(imputation_results, feature_names=adata_target.var_names)

_ = plot_imputation_metric_boxplot(imputation_results, feature_names=adata_target.var_names,
                                   feature_label="ADT protein", output_suffix="ADT", plot_type="boxplot")

Feature-wise plots complement the aggregate summary by showing how imputation accuracy varies across individual CODEX proteins.


### Representative feature recovery

A representative CODEX protein is compared with its pre-masking spatial pattern and PRISM prediction on a common display scale; PCC, SPCC and MSE summarize feature-specific recovery.


In [ ]:
# Visualize a representative CODEX protein
feature_panel = plot_prism_imputation_spatial(imputation_results=imputation_results, split1_indices=missing_indices,
                                              feature="SMA", show_missing_only=False, highlight_missing=False,
                                              ighlight_linewidth=0.01)

feature_index = adata_target.var_names.get_loc("SMA")
feature_metrics = {metric: round(float(imputation_results["raw"]["per_protein"][metric][feature_index]), 4)
                   for metric in ("PCC", "SPCC", "MSE")}
print(f"Representative CODEX protein SMA: {feature_metrics}")